# 05. Datasets & Experiments

In this notebook, you will learn how to systematically test and improve
LLM applications using Langfuse's dataset and experiment features.

## What you will learn
- Creating datasets and adding items
- Running experiments with `run_experiment()`
- Defining task functions and evaluator functions
- Analyzing and comparing experiment results

## Why do we need datasets?

To answer the question "I changed the prompt — did it actually get better?",
you need to run repeated experiments with the same set of inputs.

```
Dataset (fixed test cases)
  ├─ Experiment A (prompt v1)
  │    ├─ Item 1 result → score
  │    └─ Item 2 result → score
  └─ Experiment B (prompt v2)
       ├─ Item 1 result → score
       └─ Item 2 result → score
→ Compare A vs B → Determine which prompt is better using data
```

## Step 1: Initial setup

In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()
os.environ["ANTHROPIC_API_KEY"] = os.environ["CLAUDE_API_KEY"]

from langfuse import get_client
from opentelemetry.instrumentation.anthropic import AnthropicInstrumentor
from anthropic import Anthropic

AnthropicInstrumentor().instrument()
langfuse = get_client()
client = Anthropic()

print("✅ Setup complete")

## Step 2: Create a dataset

In [ ]:
# Fetches the dataset if it already exists, otherwise creates a new one
dataset = langfuse.create_dataset(
    name="korean-qa-benchmark",
    description="Benchmark dataset for Q&A quality evaluation",
    metadata={"domain": "general-knowledge", "language": "ko"}
)

print(f"✅ Dataset '{dataset.name}' ready")

## Step 3: Add dataset items

Each item consists of an `input` (test input) and an `expected_output` (ground truth / expected answer).

In [ ]:
dataset_items = [
    {
        "input": {"question": "What is the capital of South Korea?"},
        "expected_output": "Seoul",
        "metadata": {"category": "geography", "difficulty": "easy"}
    },
    {
        "input": {"question": "What is the main difference between a list and a tuple in Python?"},
        "expected_output": "Lists are mutable and tuples are immutable.",
        "metadata": {"category": "programming", "difficulty": "medium"}
    },
    {
        "input": {"question": "What does HTTP status code 404 mean?"},
        "expected_output": "The requested resource was not found (Not Found)",
        "metadata": {"category": "web", "difficulty": "easy"}
    },
    {
        "input": {"question": "In Big-O notation, what algorithms have O(n log n) time complexity?"},
        "expected_output": "Efficient sorting algorithms such as Merge Sort and Heap Sort",
        "metadata": {"category": "algorithms", "difficulty": "hard"}
    },
    {
        "input": {"question": "What is the difference between INNER JOIN and LEFT JOIN in SQL?"},
        "expected_output": "INNER JOIN returns only matching rows from both tables, while LEFT JOIN returns all rows from the left table.",
        "metadata": {"category": "database", "difficulty": "medium"}
    },
]

for item in dataset_items:
    langfuse.create_dataset_item(
        dataset_name="korean-qa-benchmark",
        input=item["input"],
        expected_output=item["expected_output"],
        metadata=item["metadata"]
    )

print(f"✅ {len(dataset_items)} items added")

> **Check in the dashboard:**
> You can view the `korean-qa-benchmark` dataset under the **Datasets** menu.

## Step 4: Quick experiment with local data

`run_experiment()` can also run experiments with local data, without a Langfuse-hosted dataset.
This is the fastest way to get started with experiments.

In [ ]:
from langfuse.experiment import Evaluation

# task function: runs for each dataset item
# item is a LocalExperimentItem (TypedDict) → use dict access
def simple_qa_task(*, item, **kwargs) -> str:
    question = item["input"]["question"]
    
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=256,
        messages=[{"role": "user", "content": question}]
    )
    return response.content[0].text


# evaluator function: evaluates the task result and returns an Evaluation object
# signature: receives input, output, expected_output, metadata as individual arguments
def keyword_evaluator(*, input, output, expected_output=None, metadata=None, **kwargs) -> Evaluation:
    """Check if keywords from expected output appear in actual output"""
    expected = (expected_output or "").lower()
    actual = (output or "").lower()
    
    expected_words = [w for w in expected.split() if len(w) > 2]
    
    if not expected_words:
        return Evaluation(name="keyword_match", value=0.5, comment="No expected output")
    
    matched = sum(1 for word in expected_words if word in actual)
    score = matched / len(expected_words)
    
    return Evaluation(
        name="keyword_match",
        value=score,
        comment=f"{matched}/{len(expected_words)} keywords matched"
    )


# Run the experiment
# name: experiment name, data: list of LocalExperimentItems (reusing dataset_items defined above)
result = langfuse.run_experiment(
    name="simple-qa-v1",
    run_name="claude-sonnet-no-system-prompt",
    data=dataset_items,
    task=simple_qa_task,
    evaluators=[keyword_evaluator],
)

print("Experiment results:")
print(result.format(include_item_results=True))

## Step 5: Comparison experiment — adding a system prompt

Let's compare whether adding a better system prompt improves performance.

In [ ]:
def improved_qa_task(*, item, **kwargs) -> str:
    """Task using an improved system prompt"""
    question = item["input"]["question"]
    
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=256,
        system="""You are a professional AI assistant that provides accurate and concise answers.
- Answer in 1-3 sentences covering only the key points
- Skip unnecessary introductions or elaborations
- Respond in English""",
        messages=[{"role": "user", "content": question}]
    )
    return response.content[0].text


result_v2 = langfuse.run_experiment(
    name="simple-qa-v2-with-system-prompt",
    run_name="claude-sonnet-with-system-prompt",
    data=dataset_items,
    task=improved_qa_task,
    evaluators=[keyword_evaluator],
)

print("Improved experiment results:")
print(result_v2.format(include_item_results=True))

## Step 6: Using an LLM-as-a-Judge evaluator

Instead of keyword matching, we use Claude itself as a judge for more sophisticated evaluation.

In [ ]:
def llm_judge_evaluator(*, input, output, expected_output=None, metadata=None, **kwargs) -> Evaluation:
    """Use Claude to evaluate response quality on a 0-1 scale"""
    question = input.get("question", "") if isinstance(input, dict) else str(input)
    expected = expected_output or ""
    
    if not output:
        return Evaluation(name="llm_judge_score", value=0.0, comment="No output")
    
    judge_prompt = f"""Please evaluate the following AI response to the given question.

Question: {question}
Expected answer (for reference): {expected}
Actual AI response: {output}

Scoring criteria:
- 1.0: Perfect answer (accurate, complete, and clear)
- 0.7: Good answer (covers key points, slightly lacking)
- 0.5: Average answer (partially correct)
- 0.3: Poor answer (missing key points)
- 0.0: Wrong answer (factual error)

You must respond with only the following JSON: {{"score": <0.0-1.0>, "reason": "<one sentence explanation>"}}"""
    
    try:
        judge_result = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=128,
            messages=[{"role": "user", "content": judge_prompt}]
        )
        raw = judge_result.content[0].text
        start = raw.find("{")
        end = raw.rfind("}") + 1
        parsed = json.loads(raw[start:end])
        return Evaluation(
            name="llm_judge_score",
            value=float(parsed["score"]),
            comment=parsed.get("reason", "")
        )
    except Exception as e:
        return Evaluation(name="llm_judge_score", value=0.5, comment=f"Evaluation failed: {e}")


result_v3 = langfuse.run_experiment(
    name="simple-qa-llm-judge",
    run_name="claude-sonnet-llm-judge-eval",
    data=dataset_items,
    task=improved_qa_task,
    evaluators=[keyword_evaluator, llm_judge_evaluator],
)

print("LLM-as-a-Judge evaluation results:")
print(result_v3.format(include_item_results=True))

> **Compare in the dashboard:**
> Go to **Datasets** → `korean-qa-benchmark` → **Runs** tab.
> Select multiple experiment runs side by side to compare scores for the same dataset at a glance.

## Step 7: Retrieve dataset items

In [ ]:
dataset = langfuse.get_dataset("korean-qa-benchmark")

print(f"Dataset: {dataset.name}")
print(f"Description: {dataset.description}")
print(f"Items: {len(dataset.items)}")
print()

for i, item in enumerate(dataset.items, 1):
    print(f"{i}. {item.input['question'][:50]}...")
    print(f"   Expected: {item.expected_output}")
    print(f"   Category: {item.metadata.get('category', 'N/A')}")

## Step 8: Complete workflow summary

Here is everything we have learned, organized into a single workflow.

In [ ]:
print("""
=== Langfuse Experiment Workflow ===

1. Prepare a dataset (local list or Langfuse-hosted)
   local_data = [{"input": {...}, "expected_output": "..."}]
   # or
   langfuse.create_dataset(name="my-dataset")
   langfuse.create_dataset_item(dataset_name="my-dataset", input={...}, expected_output="...")
   dataset = langfuse.get_dataset("my-dataset")

2. Define a task function (LLM call)
   def my_task(*, item, **kwargs) -> str:
       # Local data: item["input"] (dict access)
       # Langfuse dataset: item.input  (attribute access)
       return llm_call(item["input"])

3. Define an evaluator function (quality measurement)
   def my_evaluator(*, item, output, **kwargs) -> dict:
       return {"score_name": 0.0~1.0}

4. Run the experiment
   langfuse.run_experiment(
       name="experiment name",       # not experiment_name
       run_name="run identifier",
       data=local_data,              # not dataset_name, pass the actual list
       task=my_task,
       evaluators=[my_evaluator]
   )

5. Compare in the dashboard
   Datasets → select a dataset → Runs tab for A/B comparison
""")

## Full tutorial summary

| Notebook | Topic | Key APIs |
|----------|-------|----------|
| 01 | Basic tracing | `get_client()`, `AnthropicInstrumentor()`, `flush()` |
| 02 | Decorators & nesting | `@observe()`, `propagate_attributes()` |
| 03 | Prompt management | `create_prompt()`, `get_prompt()`, `compile()` |
| 04 | Evaluation & scoring | `create_score()`, LLM-as-a-Judge |
| 05 | Datasets & experiments | `create_dataset()`, `run_experiment()` |

## Next steps

- [Langfuse official docs](https://langfuse.com/docs) - Explore more features
- [LangChain integration](https://langfuse.com/docs/integrations/langchain) - Use with LangChain
- [Self-hosting](https://langfuse.com/self-hosting) - When you want to manage your data yourself